[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C17_Classical_NLP_Course/02_ngram_lm/02_ngram_lm.ipynb)

# 02 · N-gram 语言模型（纯 numpy 从零）

目标：**不调任何 LM 库**，从零实现 n-gram 计数、**Laplace / Good-Turing / Kneser-Ney 平滑**、**困惑度**、**回退/插值**，并验证每个平滑下**概率严格归一化**、KN 的困惑度低于 Laplace。

路线：句子加 `<s></s>` + n-gram 计数 → MLE 的零概率灾难 → Laplace → Good-Turing → **Kneser-Ney（绝对折扣 + 续延概率）** → 困惑度 → ✏️ 练习 → 📖 答案 → 🧪 真实语料胶囊。

> 心智模型：**数频率得到概率，平滑把零概率补成非零，困惑度 = 模型每步面临的有效分支数**。

## 1 · n-gram 计数与 MLE

给每句话加句首 `<s>`、句尾 `</s>`，然后数 unigram / bigram。bigram 的 MLE 概率 = $\frac{c(w_{t-1}, w_t)}{c(w_{t-1})}$。

In [ ]:
import numpy as np
from collections import Counter, defaultdict

def add_markers(sent_tokens):
    return ['<s>'] + list(sent_tokens) + ['</s>']

corpus_sents = [
    'the cat sat on the mat'.split(),
    'the dog sat on the log'.split(),
    'the cat chased the dog'.split(),
    'a dog sat on a mat'.split(),
]
marked = [add_markers(s) for s in corpus_sents]

def count_ngrams(marked_sents):
    uni, bi = Counter(), Counter()
    for s in marked_sents:
        uni.update(s)
        bi.update(zip(s[:-1], s[1:]))
    return uni, bi

uni, bi = count_ngrams(marked)
print('unigram(the) =', uni['the'], '| bigram(the,cat) =', bi[('the', 'cat')])

def mle_bigram(w_prev, w, uni, bi):
    if uni[w_prev] == 0:
        return 0.0
    return bi[(w_prev, w)] / uni[w_prev]

p_cat = mle_bigram('the', 'cat', uni, bi)
print('P_MLE(cat | the) =', round(p_cat, 3))
assert abs(p_cat - bi[('the', 'cat')] / uni['the']) < 1e-12
# 给定上下文 the，所有后继的 MLE 概率应和为 1
succ = [w for (p, w) in bi if p == 'the']
total = sum(mle_bigram('the', w, uni, bi) for w in succ)
assert abs(total - 1.0) < 1e-9, 'MLE 在见过的上下文下应归一'
print('✅ n-gram 计数 + MLE：见过的上下文下概率归一')

## 2 · MLE 的零概率灾难

MLE 给任何没见过的 bigram 概率 0。只要测试句含一个未见 bigram，**整句概率为 0、对数似然为 $-\infty$、困惑度为 $\infty$**。亲眼看一次这个灾难。

In [ ]:
def sentence_logprob_mle(sent, uni, bi):
    s = add_markers(sent)
    logp = 0.0
    for w_prev, w in zip(s[:-1], s[1:]):
        p = mle_bigram(w_prev, w, uni, bi)
        if p == 0.0:
            return float('-inf'), (w_prev, w)
        logp += np.log(p)
    return logp, None

# 测试句含训练中没出现的 bigram (cat, ran)
test = 'the cat ran'.split()
lp, culprit = sentence_logprob_mle(test, uni, bi)
print(f'P_MLE({test}) 的 log = {lp}  (罪魁未见 bigram: {culprit})')
assert lp == float('-inf'), 'MLE 遇未见 bigram 应给 -inf'
assert culprit == ('cat', 'ran')
print('✅ 零概率灾难重现：一个未见 bigram 让整句概率归零 -> 必须平滑')

## 3 · Laplace（加一）平滑

给每个 bigram 计数 +1，分母 +V（V 为词表大小）保持归一：
$P(w_t|w_{t-1}) = \frac{c(w_{t-1},w_t)+1}{c(w_{t-1})+V}$。它能消除零概率，但削峰过狠。

In [ ]:
vocab = sorted(uni.keys())
V = len(vocab)
print('词表(含<s></s>):', V, '个')

def laplace_bigram(w_prev, w, uni, bi, V, delta=1.0):
    return (bi[(w_prev, w)] + delta) / (uni[w_prev] + delta * V)

# 未见 bigram 现在有非零概率
p_ran = laplace_bigram('cat', 'ran', uni, bi, V)
print('P_Laplace(ran | cat) =', round(p_ran, 4), '(不再是 0)')
assert p_ran > 0
# 归一性：给定上下文 'the'，对整个词表求和应为 1
tot = sum(laplace_bigram('the', w, uni, bi, V) for w in vocab)
print('Σ_w P_Laplace(w | the) =', round(tot, 6))
assert abs(tot - 1.0) < 1e-9, 'Laplace 必须在全词表上归一'
print('✅ Laplace 平滑：未见 bigram 获非零概率，且对全词表归一')

## 4 · Good-Turing：用频次的频次重估

Good-Turing 把出现 $r$ 次的事件的计数调整为 $r^* = (r+1)\frac{N_{r+1}}{N_r}$，其中 $N_r$ 是「恰好出现 $r$ 次的 n-gram 种类数」。它的灵魂：**用「只出现一次的词有多少」估计「没出现的词有多少」**。

In [ ]:
def freq_of_freqs(bi):
    '''N_r = 恰好出现 r 次的 bigram 种类数。'''
    Nr = Counter(bi.values())
    return Nr

Nr = freq_of_freqs(bi)
print('频次的频次 N_r:', dict(sorted(Nr.items())))

def good_turing_star(r, Nr):
    '''调整计数 r* = (r+1) N_{r+1}/N_r ；缺失则退回 r。'''
    if Nr.get(r, 0) == 0 or Nr.get(r + 1, 0) == 0:
        return float(r)
    return (r + 1) * Nr[r + 1] / Nr[r]

r1_star = good_turing_star(1, Nr)
print(f'出现 1 次的 bigram 调整后计数 r* = {r1_star:.3f} (< 1, 把质量让给未见事件)')
# 未见事件(出现 0 次)的总概率估计 = N_1 / 总 bigram 数
total_bigrams = sum(bi.values())
p0_mass = Nr[1] / total_bigrams
print(f'分给所有未见 bigram 的概率质量 ≈ N_1/N = {p0_mass:.3f}')
assert r1_star < 1.0, 'GT 应把出现 1 次的计数下调'
assert 0 < p0_mass < 1
print('✅ Good-Turing：用 N_1 估计未见质量，是 Kneser-Ney 折扣的思想前身')

## 5 · Kneser-Ney：绝对折扣 + 续延概率

KN 的两大创新：
1. **绝对折扣**：每个非零计数减去固定 $d$，$\frac{\max(c(w',w)-d,0)}{c(w')}$。
2. **续延概率**：低阶用 $P_{\text{cont}}(w)=\frac{|\{w':c(w',w)>0\}|}{|\text{不同 bigram 总数}|}$（跟在多少种词后面），而非词频。

组合：$P_{KN}(w|w') = \frac{\max(c(w',w)-d,0)}{c(w')} + \lambda(w')\,P_{\text{cont}}(w)$，$\lambda(w')=\frac{d}{c(w')}\cdot|\{w:c(w',w)>0\}|$（把折扣出的质量归一的回退权重）。

In [ ]:
def kneser_ney(marked_sents, d=0.75):
    uni, bi = count_ngrams(marked_sents)
    vocab = sorted(uni.keys())
    # 续延计数：每个词前面出现过的不同词种数
    preceding = defaultdict(set)
    for (w_prev, w) in bi:
        preceding[w].add(w_prev)
    n_distinct_bigrams = len(bi)
    p_cont = {w: len(preceding[w]) / n_distinct_bigrams for w in vocab}
    # 每个上下文 w' 后面出现过的不同词种数 (用于回退权重)
    following = defaultdict(set)
    for (w_prev, w) in bi:
        following[w_prev].add(w)

    def prob(w_prev, w):
        c_prev = uni[w_prev]
        if c_prev == 0:
            return p_cont.get(w, 1.0 / len(vocab))
        first = max(bi[(w_prev, w)] - d, 0.0) / c_prev
        lam = (d / c_prev) * len(following[w_prev])
        return first + lam * p_cont.get(w, 0.0)
    return prob, vocab, p_cont

kn_prob, vocab, p_cont = kneser_ney(marked, d=0.75)
print('P_KN(cat | the) =', round(kn_prob('the', 'cat'), 4))
# ('cat','log') 在训练里没出现，但 'log' 这个词见过(在 'the log') -> 续延概率>0 -> 回退非零
print('P_KN(log | cat) =', round(kn_prob('cat', 'log'), 4), '(未见 bigram, 但词已知 -> 非零)')
assert kn_prob('cat', 'log') > 0, '未见 bigram 但词已知, KN 应回退到正的续延概率'
# 归一性：KN 在每个上下文下也必须归一
for ctx in ['the', 'cat', 'a']:
    tot = sum(kn_prob(ctx, w) for w in vocab)
    assert abs(tot - 1.0) < 1e-9, f'KN 在上下文 {ctx} 下应归一, 得 {tot}'
print('✅ Kneser-Ney：折扣+续延概率，在每个上下文下严格归一')
print('   注：完全没见过的词(OOV) 续延概率为 0 —— 这要靠 <unk> 处理，见小结')

## 6 · 困惑度：比较 Laplace vs Kneser-Ney

$\mathrm{PPL} = \exp(-\frac1N\sum_t \log P(w_t|w_{t-1}))$。在**对数空间**累加（防下溢）。在留出测试句上比较两种平滑——KN 应给出更低的困惑度。

In [ ]:
def perplexity(test_sents, prob_fn):
    '''prob_fn(w_prev, w) -> 概率。返回测试集困惑度。'''
    total_logp, N = 0.0, 0
    for sent in test_sents:
        s = add_markers(sent)
        for w_prev, w in zip(s[:-1], s[1:]):
            p = prob_fn(w_prev, w)
            total_logp += np.log(max(p, 1e-12))
            N += 1
    return float(np.exp(-total_logp / N))

test_sents = [
    'the cat sat on the log'.split(),    # 见过的词，新组合
    'the dog chased a cat'.split(),
]
laplace_fn = lambda wp, w: laplace_bigram(wp, w, uni, bi, V)
ppl_lap = perplexity(test_sents, laplace_fn)
ppl_kn = perplexity(test_sents, kn_prob)
print(f'困惑度  Laplace = {ppl_lap:.2f}')
print(f'困惑度  Kneser-Ney = {ppl_kn:.2f}')
assert ppl_kn < ppl_lap, 'KN 困惑度应低于 Laplace'
assert ppl_kn > 1.0, '困惑度必 >= 1'
print('✅ Kneser-Ney 困惑度低于 Laplace —— 平滑质量的体现')

---
## ✏️ 练习 1：bigram 计数与 MLE

实现 `bigram_mle(w_prev, w, uni, bi)`：返回 $\frac{c(w_{t-1},w_t)}{c(w_{t-1})}$，上下文未见时返回 0。

In [ ]:
def bigram_mle(w_prev, w, uni, bi):
    # TODO: 返回 bi[(w_prev,w)] / uni[w_prev]，分母为 0 时返回 0.0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
u, b = count_ngrams(marked)
assert abs(bigram_mle('the', 'cat', u, b) - b[('the','cat')]/u['the']) < 1e-12
assert bigram_mle('zzz', 'cat', u, b) == 0.0, '未见上下文应返回 0'
succ = set(w for (pp, w) in b if pp == 'on')
assert abs(sum(bigram_mle('on', w, u, b) for w in succ) - 1.0) < 1e-9
print('✅ 练习 1 通过：bigram MLE 正确且在见过的上下文下归一')

## ✏️ 练习 2：Laplace 平滑与归一

实现 `laplace(w_prev, w, uni, bi, V, delta)`：$\frac{c+\delta}{c(w_{t-1})+\delta V}$，并保证它在整个词表上归一化为 1。

In [ ]:
def laplace(w_prev, w, uni, bi, V, delta=1.0):
    # TODO: (bi[(w_prev,w)] + delta) / (uni[w_prev] + delta*V)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
vocab2 = sorted(u.keys()); V2 = len(vocab2)
assert laplace('cat', 'ran', u, b, V2) > 0, '未见 bigram 应非零'
for ctx in ['the', 'cat', '<s>']:
    tot = sum(laplace(ctx, w, u, b, V2) for w in vocab2)
    assert abs(tot - 1.0) < 1e-9, f'{ctx} 下未归一: {tot}'
# delta 越大越平(接近均匀) -> 未见 bigram 概率越大
assert laplace('cat','ran',u,b,V2,delta=2.0) > laplace('cat','ran',u,b,V2,delta=0.5)
print('✅ 练习 2 通过：Laplace 平滑非零且归一')

## ✏️ 练习 3：续延概率（Kneser-Ney 核心）

实现 `continuation_prob(bi)`：返回 dict `{w: P_cont(w)}`，$P_{\text{cont}}(w) = \frac{|\{w': c(w',w)>0\}|}{|\text{不同 bigram 总数}|}$（每个词跟在多少种不同词后面 / 不同 bigram 数）。

In [ ]:
from collections import defaultdict
def continuation_prob(bi):
    # TODO: 统计每个 w 前面出现过的不同 w' 种数，除以不同 bigram 总数
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
pc = continuation_prob(b)
assert abs(sum(pc.values()) - 1.0) < 1e-9, '续延概率应是合法分布(和为1)'
# 'sat' 跟在 cat/dog 后面(>=2 种) 应有正续延概率
assert pc['sat'] > 0
# 构造对照：只跟在一种词后的低续延 vs 跟在多种词后的高续延
toy = Counter({('a','x'):5, ('a','y'):1, ('b','y'):1, ('c','y'):1})
pc2 = continuation_prob(toy)
# x 只跟在 a 后(1种); y 跟在 a,b,c 后(3种) -> y 续延概率更高
assert pc2['y'] > pc2['x'], 'y 跟在更多种词后, 续延概率应更高'
print('✅ 练习 3 通过：续延概率用语境多样性而非频次')

## ✏️ 练习 4：困惑度

实现 `perplexity(test_sents, prob_fn)`：$\exp(-\frac1N\sum\log P)$，**在对数空间累加**（防下溢）。$N$ 为测试集 bigram 总数。

In [ ]:
def perplexity(test_sents, prob_fn):
    # TODO: 对每句加 <s></s>, 累加 log P(w|w_prev), 返回 exp(-总和/N)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
kn_fn, vv, _ = kneser_ney(marked, d=0.75)
ppl = perplexity([['the','cat','sat']], kn_fn)
assert ppl > 1.0, '困惑度 >= 1'
# 均匀分布(1/V)的困惑度应约等于 V
uniform_fn = lambda wp, w: 1.0 / V2
ppl_uni = perplexity([['the','cat','sat']], uniform_fn)
assert abs(ppl_uni - V2) < 1e-6, f'均匀模型困惑度应=V={V2}, 得 {ppl_uni}'
# 更好的模型(KN) 困惑度应低于均匀
assert ppl < ppl_uni
print(f'✅ 练习 4 通过：KN PPL={ppl:.1f} < 均匀 PPL={ppl_uni:.1f}')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def bigram_mle(w_prev, w, uni, bi):
    if uni[w_prev] == 0:
        return 0.0
    return bi[(w_prev, w)] / uni[w_prev]

In [ ]:
# 练习 2 参考答案
def laplace(w_prev, w, uni, bi, V, delta=1.0):
    return (bi[(w_prev, w)] + delta) / (uni[w_prev] + delta * V)

In [ ]:
# 练习 3 参考答案
def continuation_prob(bi):
    preceding = defaultdict(set)
    for (w_prev, w) in bi:
        preceding[w].add(w_prev)
    n = len(bi)
    return {w: len(s) / n for w, s in preceding.items()}

In [ ]:
# 练习 4 参考答案
def perplexity(test_sents, prob_fn):
    total_logp, N = 0.0, 0
    for sent in test_sents:
        s = add_markers(sent)
        for w_prev, w in zip(s[:-1], s[1:]):
            total_logp += np.log(max(prob_fn(w_prev, w), 1e-12))
            N += 1
    return float(np.exp(-total_logp / N))

---
## 🧪 真实数据胶囊：在真实文本上训 n-gram LM

用真实英文文本（**下载 tiny-shakespeare，失败回退内置真实片段**）训练 bigram KN 模型，在留出句上算困惑度，并对比 Laplace。

In [ ]:
def load_sentences():
    '''下载 tiny-shakespeare；失败回退内置真实片段。返回句子列表(每句 token 列表)。'''
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    try:
        import urllib.request
        with urllib.request.urlopen(url, timeout=5) as r:
            text = r.read().decode('utf-8')
        src = 'downloaded'
    except Exception:
        text = ('To be or not to be that is the question. '
                'Whether tis nobler in the mind to suffer. '
                'The slings and arrows of outrageous fortune. '
                'Or to take arms against a sea of troubles. '
                'And by opposing end them to die to sleep. '
                'No more and by a sleep to say we end. ') * 6
        src = 'builtin fallback'
    import re
    # 按句号切句、小写、按非字母切词；只取前若干句(玩具规模)
    raw_sents = re.split(r'[.!?]', text.lower())[:120]
    sents = [re.findall(r"[a-z']+", s) for s in raw_sents]
    sents = [s for s in sents if 2 <= len(s) <= 20]
    return sents, src

sents, src = load_sentences()
print('数据来源:', src, '| 句数:', len(sents))
split = max(1, int(len(sents) * 0.8))
train_raw, test_raw = sents[:split], sents[split:]

# OOV 处理(标准做法, 见模块讲解第 7 节):
#  ① 训练集里只出现 1 次的词 -> <unk>, 让 <unk> 获得计数与续延概率
#  ② 测试集里不在训练词表的词 -> <unk>
wc = Counter(w for s in train_raw for w in s)
def unkify(sent, keep):
    return [w if w in keep else '<unk>' for w in sent]
keep_vocab = {w for w, c in wc.items() if c >= 2}     # 出现>=2次的词保留
train_sents = [unkify(s, keep_vocab) for s in train_raw]
test_sents_real = [unkify(s, keep_vocab) for s in test_raw]
print('训练句:', len(train_sents), '| 测试句:', len(test_sents_real),
      '| 保留词表:', len(keep_vocab), '(+<unk>)')
assert len(train_sents) > 0 and len(test_sents_real) > 0
assert any('<unk>' in s for s in train_sents), '训练集应有 <unk>(来自低频词)'
print('✅ 真实语料就绪(已用 <unk> 处理 OOV —— 这样 KN 才能正确回退)')

**🧪 胶囊练习**：把训练句加 marker、训练 KN（`kneser_ney`），在测试句上算困惑度。补全两行。

In [ ]:
marked_train = [add_markers(s) for s in train_sents]
# TODO: kn_real, voc_real, _ = kneser_ney(marked_train, d=0.75)
#       ppl_real = perplexity(test_sents_real, kn_real)
raise NotImplementedError

In [ ]:
# 自测
assert ppl_real > 1.0, '困惑度 >= 1'
uni_r, bi_r = count_ngrams(marked_train)
V_r = len(uni_r)
lap_real = lambda wp, w: laplace(wp, w, uni_r, bi_r, V_r)
ppl_lap_real = perplexity(test_sents_real, lap_real)
print(f'真实语料: KN PPL={ppl_real:.1f}  Laplace PPL={ppl_lap_real:.1f}')
assert ppl_real < ppl_lap_real, 'KN 应优于 Laplace'
print('✅ 胶囊练习通过：真实文本上 KN 困惑度低于 Laplace')

In [ ]:
# 📖 胶囊参考答案
marked_train = [add_markers(s) for s in train_sents]
kn_real, voc_real, _ = kneser_ney(marked_train, d=0.75)
ppl_real = perplexity(test_sents_real, kn_real)
print('KN 困惑度(真实语料) =', round(ppl_real, 2))

### 小结
- **语言模型** = 给词序列赋概率 = 预测下一个词。链式法则精确，马尔可夫假设近似。
- **n-gram MLE** 数频率，但给未见 n-gram 概率 0（**零概率灾难**）。
- **平滑** 劫富济贫：Laplace(加一,削峰狠) < Good-Turing(频次的频次) < **Kneser-Ney**(绝对折扣+续延概率)。
- **续延概率** 是 KN 精髓：用「跟在多少种不同词后面」而非词频估计低阶 —— Francisco 高频但续延低。
- **困惑度** $\exp(-\frac1N\sum\log P)$ = 每步有效分支数，越低越好；在对数空间算、在测试集上算。
- **回退**(用一个) vs **插值**(混合全部)；KN 本质是插值平滑。

下一站：**模块 03 · 隐马尔可夫模型** —— 给序列的每个词标注隐藏标签（如词性）。